[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/41_cosine_lr.ipynb)

# 🟡 Medium: Cosine LR Scheduler with Warmup

*Training*
Implement the learning-rate schedule that essentially every modern transformer
is trained with: **linear warmup, then cosine decay**.

$$
\eta(t) =
\begin{cases}
\eta_{\max}\dfrac{t}{T_w} & t < T_w \\[2ex]
\eta_{\min} + \tfrac12(\eta_{\max}-\eta_{\min})
\left(1 + \cos\left(\pi\dfrac{t-T_w}{T-T_w}\right)\right) & t \ge T_w
\end{cases}
$$

### Signature
```python
def cosine_schedule(step, base_lr, warmup_steps, total_steps, min_lr=0.0):
    ...  # -> learning rate at `step`
```

### Rules
- `step` may be a **scalar or an array** of steps — the function must be
  vectorised, so branch with `jnp.where`, not `if`
- Must be `jax.jit`-able with `step` traced
- Clamp past the end: for `step >= total_steps` the rate stays at `min_lr`
- Do not use `optax.warmup_cosine_decay_schedule`

### Boundary conventions this is graded on
- $\eta(0) = 0$ when $T_w > 0$ (and $\eta_{\max}$ when $T_w = 0$ — the ramp is empty)
- $\eta(T_w) = \eta_{\max}$ exactly — warmup ends *at* the peak
- $\eta(T) = \eta_{\min}$ exactly
- the halfway point of decay is $(\eta_{\max}+\eta_{\min})/2$

The two branches are chosen so that they **agree at the seam**: warmup's
$\eta_{\max} t / T_w$ hits $\eta_{\max}$ at $t = T_w$, and the cosine at
progress 0 is also $\eta_{\max}$, so it does not matter whether the comparison
is `<` or `<=` and the schedule is continuous either way. The off-by-one that
*does* bite is the warmup numerator: writing `(step + 1) / T_w` (or dividing by
`T_w - 1`) gives $\eta(0) \neq 0$ or overshoots the peak.

### Why warmup exists
At step 0 Adam's second-moment estimate $v$ has seen exactly one gradient, so
$\hat{m}/\sqrt{\hat{v}}$ is an unreliable *direction* whose magnitude is pinned
near 1 — full-size steps along a badly-estimated direction is how early training
diverges, and the deeper the network the worse it is. Warmup buys the moment
estimates time to become meaningful. That is also why [[adam]]'s bias correction
and warmup are usually discussed together (get the correction wrong and the
early steps are several times *too large*, which warmup then partially hides),
and why architectures that stabilise early gradients (pre-norm) need much less
warmup.

### Why cosine rather than linear
Both start at $\eta_{\max}$ and end at $\eta_{\min}$; the difference is how the
budget in between is spent. Cosine is **flatter at both ends and steeper in the
middle**: at 25% through decay it is still at $0.854\,\eta_{\max}$ where a linear
ramp is already down to $0.75$, and at 75% it is at $0.146$ where linear is still
at $0.25$. So cosine holds a near-peak rate through the early part of decay —
where most of the learning happens — and then anneals hard.

Its derivative $-\tfrac{\pi}{2}(\eta_{\max}-\eta_{\min})\sin(\pi p)$ vanishes at
both $p = 0$ and $p = 1$, so the rate leaves the peak and arrives at the floor
smoothly, with none of the loss spikes that step decay's discontinuities cause.
The cost is that the whole shape is pinned to $T$: stop early and you stop
mid-decay at a high rate, extend $T$ and every previous step was on the wrong
curve. That non-resumability is the standard interview follow-up, and it is
exactly what constant-then-decay ("WSD"-style) schedules were introduced to fix.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def cosine_schedule(step, base_lr, warmup_steps, total_steps, min_lr=0.0):
    """Learning rate at `step` — linear warmup then cosine decay.

    Args:
        step:         scalar or array of step indices
        base_lr:      peak learning rate, reached at step == warmup_steps
        warmup_steps: length of the linear ramp
        total_steps:  step at which the rate reaches min_lr
        min_lr:       floor

    Returns:
        Learning rate(s), same shape as `step`.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax.numpy as jnp

steps = jnp.arange(0, 1001, 100)
lrs = cosine_schedule(steps, base_lr=1e-3, warmup_steps=100, total_steps=1000)
for s, lr in zip(steps.tolist(), lrs.tolist()):
    bar = "█" * int(lr / 1e-3 * 40)
    print(f"{s:>5}  {lr:.6f}  {bar}")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("cosine_lr")

# hint("cosine_lr")      # stuck? nudge without the answer
# solution("cosine_lr")  # spoiler: the reference implementation